In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
import numpy as np
import time
import logging
import sys
import json
from datetime import datetime

In [2]:
# --- 1. 配置与环境设置 ---
def setup_logging(log_file='training_online_news_cofrnet_d.log'):
    log_formatter = logging.Formatter("%(asctime)s [%(levelname)-5.5s]  %(message)s")
    root_logger = logging.getLogger(); root_logger.handlers = []; root_logger.setLevel(logging.INFO)
    file_handler = logging.FileHandler(log_file, mode='w'); file_handler.setFormatter(log_formatter); root_logger.addHandler(file_handler)
    console_handler = logging.StreamHandler(sys.stdout); console_handler.setFormatter(log_formatter); root_logger.addHandler(console_handler)

torch.manual_seed(42); np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 2. 数据准备 ---
def get_regression_data_loaders(batch_size=128, test_size=0.3):
    """准备在线新闻流行度回归数据集"""
    logging.info("正在准备在线新闻流行度回归数据集...")
    
    # 加载在线新闻流行度数据集
    import pandas as pd
    df = pd.read_csv('/kaggle/input/online-news-popularity/OnlineNewsPopularity.csv')
    
    # 删除无用和共线性特征
    cols_to_drop = ['url', ' timedelta', 
                   ' abs_title_sentiment_polarity', ' LDA_04', ' is_weekend',
                   ' rate_positive_words', ' abs_title_subjectivity', 
                   ' rate_negative_words']
    df = df.drop(cols_to_drop, axis=1)
    
    # 特征和目标变量
    X = df.drop([' shares'], axis=1)  # 特征矩阵
    y = df[' shares']  # 目标变量（文章分享次数）
    
    # 数据标准化
    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()
    X = scaler.fit_transform(X)
    
    # 对目标变量取对数并标准化（因为分享次数通常呈长尾分布）
    y = np.log1p(y)  # 使用log(1+y)转换
    y = (y - y.mean()) / y.std()  # 标准化目标变量
    
    # 划分训练测试集
    from sklearn.model_selection import train_test_split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42)
    
    # 转换为PyTorch张量
    X_train = torch.FloatTensor(X_train)
    y_train = torch.FloatTensor(y_train.values)  
    X_test = torch.FloatTensor(X_test)
    y_test = torch.FloatTensor(y_test.values)   
   
    # 创建DataLoader
    train_dataset = torch.utils.data.TensorDataset(X_train, y_train)
    test_dataset = torch.utils.data.TensorDataset(X_test, y_test)
    
    trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    testloader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    logging.info("在线新闻流行度回归数据集准备完毕。")
    logging.info(f"训练样本数: {len(train_dataset)}, 测试样本数: {len(test_dataset)}")
    logging.info(f"特征维度: {X_train.shape[1]}, 目标维度: 1")
    
    return trainloader, testloader, X_train.shape[1]

# --- 3. 定义CoFrNet-D回归模型 ---
class SingleLadder(nn.Module):
    def __init__(self, depth, epsilon=0.1):
        super(SingleLadder, self).__init__()
        self.depth = depth
        self.epsilon = epsilon
        
        # 可学习的权重
        self.weights = nn.Parameter(torch.randn(depth + 1) * 0.1)
        
    def safe_reciprocal(self, x):
        sign = torch.sign(x)
        abs_x = torch.abs(x)
        safe_abs = torch.clamp(abs_x, min=self.epsilon)
        return sign / safe_abs
    
    def forward(self, x_j):
        batch_size = x_j.shape[0]
        
        if self.depth == 0:
            return self.weights[0] * x_j
        
        # 从最深层开始
        result = self.weights[self.depth] * x_j
        
        for k in range(self.depth - 1, 0, -1):
            denominator = self.weights[k] * x_j + self.safe_reciprocal(result)
            result = denominator
        
        # 最后一步: w0 * x_j + 1/result
        if self.depth > 0:
            final_result = self.weights[0] * x_j + self.safe_reciprocal(result)
        else:
            final_result = self.weights[0] * x_j
            
        return final_result

class CoFrNetDRegression(nn.Module):
    def __init__(self, input_dim, max_depth=30, epsilon=0.1):
        super(CoFrNetDRegression, self).__init__()
        self.input_dim = input_dim
        self.max_depth = max_depth
        self.epsilon = epsilon
        
        # 为每个输入维度创建一个梯形
        self.ladders = nn.ModuleList()
        self.ladder_depths = []
        
        for i in range(input_dim):
            depth = min(max_depth, max(1, 5 + i % 10))  # 可调
            self.ladder_depths.append(depth)
            
            # 单个梯形
            ladder = SingleLadder(depth, epsilon)
            self.ladders.append(ladder)
        
        # 最终层 (回归任务输出维度为1)
        self.output_layer = nn.Linear(input_dim, 1)
        
        nn.init.xavier_uniform_(self.output_layer.weight)
        nn.init.zeros_(self.output_layer.bias)
    
    def forward(self, x):
        batch_size = x.shape[0]
        ladder_outputs = []
        
        # 计算梯形
        for j in range(self.input_dim):
            x_j = x[:, j]  
            ladder_output = self.ladders[j](x_j)
            ladder_outputs.append(ladder_output.unsqueeze(1))
        
        # 连接所有梯形输出
        ladder_features = torch.cat(ladder_outputs, dim=1)  
        
        # 最终线性组合
        output = self.output_layer(ladder_features)
        
        return output.squeeze()  # 回归任务输出一维
    
    def get_feature_importance(self, x):
        device = next(self.parameters()).device  
        
        with torch.no_grad():
            if not isinstance(x, torch.Tensor):
                x = torch.FloatTensor(x)
            x = x.to(device)
            
            ladder_outputs = []
            for j in range(self.input_dim):
                x_j = x[:, j]
                ladder_output = self.ladders[j](x_j)
                # 使用平均绝对值作为重要性度量
                importance = torch.mean(torch.abs(ladder_output)).item()
                ladder_outputs.append(importance)
            
            return np.array(ladder_outputs)
    
    def get_ladder_weights(self):
        all_weights = []
        for ladder in self.ladders:
            all_weights.append(ladder.weights.detach().cpu().numpy())
        return all_weights

# --- 4. 训练函数 ---
def train_regression(model, trainloader, testloader, hparams, log_filepath, model_save_path):
    criterion = nn.MSELoss()  # 回归任务使用MSE损失
    optimizer = optim.Adam(model.parameters(), 
                         lr=hparams['learning_rate'], 
                         weight_decay=hparams['weight_decay'])
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', 
                                                   patience=hparams['patience']//3, 
                                                   factor=0.5)
    
    best_test_loss = float('inf')
    patience_counter = 0
    train_losses = []
    test_losses = []
    
    log_data = {
        "experiment_timestamp": datetime.now().isoformat(),
        "hyperparameters": hparams,
        "training_history": []
    }
    
    logging.info("="*30)
    logging.info("  启动 CoFrNet-D 回归模型训练")
    logging.info("="*30)
    
    for epoch in range(hparams['num_epochs']):
        model.train()
        running_loss = 0.0
        
        # 训练阶段
        for inputs, targets in trainloader:
            inputs, targets = inputs.to(device), targets.float().to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            
            # 梯度裁剪
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            running_loss += loss.item()
        
        epoch_train_loss = running_loss / len(trainloader)
        train_losses.append(epoch_train_loss)
        
        # 测试阶段
        model.eval()
        test_loss = 0.0
        with torch.no_grad():
            for inputs, targets in testloader:
                inputs, targets = inputs.to(device), targets.float().to(device)
                outputs = model(inputs)
                test_loss += criterion(outputs, targets).item()
        
        epoch_test_loss = test_loss / len(testloader)
        test_losses.append(epoch_test_loss)
        
        # 记录日志
        epoch_log = {
            "epoch": epoch + 1,
            "train_loss": epoch_train_loss,
            "test_loss": epoch_test_loss,
            "learning_rate": optimizer.param_groups[0]['lr']
        }
        log_data["training_history"].append(epoch_log)
        
        # 更新学习率
        scheduler.step(epoch_test_loss)
        
        # 打印进度
        logging.info(f"Epoch {epoch+1}/{hparams['num_epochs']} | "
                    f"Train Loss: {epoch_train_loss:.4f} | "
                    f"Test Loss: {epoch_test_loss:.4f} | "
                    f"LR: {optimizer.param_groups[0]['lr']:.2e}")
        
        # 早停检查
        if epoch_test_loss < best_test_loss:
            best_test_loss = epoch_test_loss
            patience_counter = 0
            # 保存最佳模型
            torch.save(model.state_dict(), model_save_path)
        else:
            patience_counter += 1
            if patience_counter >= hparams['patience']:
                logging.info(f"\n早停: {hparams['patience']}轮没有改进")
                break
    
    # 保存训练日志
    with open(log_filepath, 'w') as f:
        json.dump(log_data, f, indent=4)
    
    return best_test_loss

In [3]:
# --- 5. 主函数 ---
if __name__ == '__main__':
    # --- 文件与超参数定义 ---
    LOG_FILE = 'training_online_news_cofrnet_d.log'
    JSON_LOG_FILE = 'training_online_news_cofrnet_d_log.json'
    MODEL_SAVE_PATH = 'cofrnet_d_regression.pth'
    
    HYPERPARAMETERS = {
        "num_epochs": 50,
        "learning_rate": 0.001,
        "weight_decay": 1e-4,
        "patience": 20,
        "max_depth": 30,
        "epsilon": 0.1
    }

    setup_logging(LOG_FILE)
    logging.info(f"回归实验超参数: {json.dumps(HYPERPARAMETERS, indent=4)}")

    trainloader, testloader, input_dim = get_regression_data_loaders()
    
    # 创建CoFrNet-D回归模型
    model = CoFrNetDRegression(
        input_dim=input_dim,
        max_depth=HYPERPARAMETERS['max_depth'],
        epsilon=HYPERPARAMETERS['epsilon']
    ).to(device)
    
    total_params = sum(p.numel() for p in model.parameters())
    logging.info(f"模型总参数数量: {total_params:,}")
    
    start_time = time.time()
    best_test_loss = train_regression(
        model, trainloader, testloader,
        hparams=HYPERPARAMETERS,
        log_filepath=JSON_LOG_FILE,
        model_save_path=MODEL_SAVE_PATH
    )
    
    total_time = time.time() - start_time
    logging.info(f"总训练时间: {total_time:.2f} 秒")
    logging.info(f"最佳测试MSE损失: {best_test_loss:.4f}")
    
    # 加载并验证保存的最佳模型
    try:
        logging.info("\n" + "="*30)
        logging.info("  开始验证已保存的最佳回归模型")
        logging.info("="*30)
        
        verification_model = CoFrNetDRegression(
            input_dim=input_dim,
            max_depth=HYPERPARAMETERS['max_depth'],
            epsilon=HYPERPARAMETERS['epsilon']
        ).to(device)
        verification_model.load_state_dict(torch.load(MODEL_SAVE_PATH))
        verification_model.eval()

        test_loss = 0.0
        criterion = nn.MSELoss()
        with torch.no_grad():
            for inputs, targets in testloader:
                inputs, targets = inputs.to(device), targets.float().to(device)
                outputs = verification_model(inputs)
                test_loss += criterion(outputs, targets).item()
        
        avg_test_loss = test_loss / len(testloader)
        logging.info(f"加载后的模型在测试集上的MSE损失: {avg_test_loss:.4f}")
        
        assert abs(avg_test_loss - best_test_loss) < 1e-3, "验证失败！"
        logging.info("验证成功！回归模型可以被正确保存和复用。")
        
        # 特征重要性分析
        logging.info("\n分析特征重要性...")
        sample_batch = next(iter(testloader))
        sample_x = sample_batch[0][:100]  # 使用100个样本计算重要性
        
        importance_scores = verification_model.get_feature_importance(sample_x)
        top_indices = np.argsort(np.abs(importance_scores))[-5:][::-1]  # 取最重要的5个特征
        
        logging.info("前5个最重要的特征:")
        for i, idx in enumerate(top_indices):
            logging.info(f"  {i+1}. 特征_{idx}: {importance_scores[idx]:.4f}")
            
    except Exception as e:
        logging.error(f"验证过程中出错: {str(e)}")

2025-08-14 11:37:24,178 [INFO ]  回归实验超参数: {
    "num_epochs": 50,
    "learning_rate": 0.001,
    "weight_decay": 0.0001,
    "patience": 20,
    "max_depth": 30,
    "epsilon": 0.1
}
2025-08-14 11:37:24,179 [INFO ]  正在准备在线新闻流行度回归数据集...
2025-08-14 11:37:24,306 [INFO ]  NumExpr defaulting to 4 threads.
2025-08-14 11:37:26,109 [INFO ]  在线新闻流行度回归数据集准备完毕。
2025-08-14 11:37:26,110 [INFO ]  训练样本数: 27750, 测试样本数: 11894
2025-08-14 11:37:26,111 [INFO ]  特征维度: 52, 目标维度: 1
2025-08-14 11:37:26,349 [INFO ]  模型总参数数量: 591
2025-08-14 11:37:29,564 [INFO ]  ==============================
2025-08-14 11:37:29,565 [INFO ]    启动 CoFrNet-D 回归模型训练
2025-08-14 11:37:29,566 [INFO ]  ==============================
2025-08-14 11:38:07,349 [INFO ]  Epoch 1/50 | Train Loss: 14.5807 | Test Loss: 3.6899 | LR: 1.00e-03
2025-08-14 11:38:44,463 [INFO ]  Epoch 2/50 | Train Loss: 2.2869 | Test Loss: 1.1032 | LR: 1.00e-03
2025-08-14 11:39:20,591 [INFO ]  Epoch 3/50 | Train Loss: 0.9949 | Test Loss: 0.9093 | LR: 1.00e-03
2025-